# 02 · Position dictionary review (step 4)

Checks the LLM-built position dictionary (`position_dict`: one row per distinct form) and the mentions derived from it
(`ad_positions`: one row per position named in an ad). Needs steps 1–4.

Counts below use **distinct ads** (the canonical printing of each ad from step 3) and leave out death register entries (step 2).

In [ ]:
import pandas as pd
from IPython.display import display
from hisrag.data import query

pd.set_option("display.max_colwidth", 120)

# Distinct ads, no death register: the base for all counts below.
BASE = """
    ad_positions p
    JOIN ads a USING (ad_id)
    JOIN ad_dups d USING (ad_id)
    JOIN ad_text t USING (ad_id)
    WHERE d.is_canonical AND NOT t.flag_death_register
"""

query("""SELECT count(*) AS forms,
                  count(*) FILTER (WHERE is_position) AS positions,
                  count(*) FILTER (WHERE NOT is_position) AS not_positions,
                  count(*) FILTER (WHERE is_position IS NULL) AS failed,
                  count(*) FILTER (WHERE confidence = 'low') AS low_confidence
           FROM position_dict""")

## Forms marked low confidence
Garbled or ambiguous forms where the model only guessed. Change `seed` for other samples.

In [ ]:
def low_confidence(n=15, seed=0):
    return query(f"""SELECT surface, count, context,
                              list_transform(entries, e -> e.term || ' → ' || e.modern || ' (' || e.category || ')') AS entries
                       FROM position_dict WHERE confidence = 'low'
                       ORDER BY hash(key || '{seed}') LIMIT {n}""")
low_confidence()

## All forms behind a lemma
Useful to spot forms that were mapped to the wrong occupation.

In [ ]:
def forms_of(lemma: str):
    return query("""SELECT d.surface, d.count, e.term, e.modern, e.gender_form, e.category, d.confidence
                      FROM position_dict d, UNNEST(d.entries) AS u(e)
                      WHERE lower(e.lemma) = lower(?) ORDER BY d.count DESC""", [lemma])
forms_of("Unterlehrer")

In [ ]:
forms_of("Koch")

## Most frequent positions per decade (distinct ads)

In [ ]:
top = query(f"""
    SELECT a.decade, p.lemma, count(DISTINCT p.ad_id) AS ads
    FROM {BASE}
    GROUP BY a.decade, p.lemma QUALIFY row_number() OVER (PARTITION BY a.decade ORDER BY ads DESC) <= 5
    ORDER BY a.decade, ads DESC""")
top.pivot_table(index="decade", columns=top.groupby("decade").cumcount() + 1,
                values="lemma", aggfunc="first").rename(columns=lambda i: f"#{i}")

## Categories per decade (share of distinct ads)
Note how strongly the Wiener Zeitung is dominated by official vacancies (teachers, administration): this is a property of the source, and answers about "the labour market" must say so.

In [ ]:
cat = query(f"""
    SELECT a.decade, p.category, count(DISTINCT p.ad_id) AS ads
    FROM {BASE}
    GROUP BY ALL""")
shares = cat.pivot_table(index="category", columns="decade", values="ads", fill_value=0)
(shares / shares.sum() * 100).round(1)

## Female share of position titles
Share of mentions whose *wording* is female (`gender_form = 'f'`), per category and decade. Only decades and categories with at least 30 mentions are shown.

In [ ]:
g = query(f"""
    SELECT a.decade, p.category,
           count(*) AS mentions,
           round(100 * avg((p.gender_form = 'f')::int), 1) AS pct_female_form
    FROM {BASE}
    GROUP BY ALL HAVING count(*) >= 30""")
g.pivot_table(index="category", columns="decade", values="pct_female_form")